# Includes

In [2]:
import pandas as pd
import numpy as np
import mne
import mne_nirs
import h5py
import shutil

from pathlib import Path
from mne.preprocessing.nirs import source_detector_distances, short_channels

root = Path.home() / "fnirs-representation-learning"
rs_data_dir = root / "snirf_dataset_2"

/home/asunkari/miniconda3/envs/neuro-ml/lib/python3.12/site-packages/mne/datasets/eegbci/eegbci.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [12]:
clean_file_rows = []

for subject_dir in sorted(rs_data_dir.glob("Subj*")):
    if not subject_dir.is_dir():
        continue

    resting_file = subject_dir / "resting.snirf"
    clean_file = subject_dir / "resting_clean.snirf"

    if not clean_file.exists():
        shutil.copy2(resting_file, clean_file)

        with h5py.File(clean_file, "r+") as f:
            if "stim1" in f["nirs"]:
                del f["nirs"]["stim1"]

        print("created:", clean_file)
        
    else:
        print("already exists:", clean_file)

    clean_file_rows.append({
        "subject": subject_dir.name,
        "clean_file": clean_file,
    })

clean_files_df = pd.DataFrame(clean_file_rows)
clean_files_df

already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj92/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj94/resting_clean.snirf
already exists: /home/asunkari/fnirs-representation-learnin

,subject,clean_file
0,Subj100,/home/asunkari/fnirs-representation-learning/s...
1,Subj101,/home/asunkari/fnirs-representation-learning/s...
2,Subj102,/home/asunkari/fnirs-representation-learning/s...
3,Subj103,/home/asunkari/fnirs-representation-learning/s...
4,Subj104,/home/asunkari/fnirs-representation-learning/s...
5,Subj86,/home/asunkari/fnirs-representation-learning/s...
6,Subj91,/home/asunkari/fnirs-representation-learning/s...
7,Subj92,/home/asunkari/fnirs-representation-learning/s...
8,Subj94,/home/asunkari/fnirs-representation-learning/s...
9,Subj95,/home/asunkari/fnirs-representation-learning/s...


In [14]:
subject_summary_rows = []
pair_rows = []

for row in clean_files_df.itertuples(index=False):
    subject = row.subject
    clean_file = row.clean_file

    raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)

    # pick all fNIRS channels, then keep only CW amplitude channels
    picks_fnirs = mne.pick_types(raw_rest.info, fnirs=True)
    channel_types = np.array(raw_rest.get_channel_types())
    picks_cw = picks_fnirs[channel_types[picks_fnirs] == "fnirs_cw_amplitude"]

    # distances and SS/LS masks
    dists = source_detector_distances(raw_rest.info, picks=picks_cw)

    ss_mask_all = short_channels(raw_rest.info, threshold=0.015)
    ss_mask = ss_mask_all[picks_cw]

    ls_mask = dists >= 0.025

    # channel-level names
    cw_names = np.array(raw_rest.ch_names)[picks_cw]
    pair_names = np.array([name.split(" ")[0] for name in cw_names])

    # pair-level table for this subject
    pair_table = pd.DataFrame({
        "subject": subject,
        "channel_name": cw_names,
        "pair_name": pair_names,
        "distance_m": dists,
        "is_ss": ss_mask,
        "is_ls": ls_mask,
    })

    pair_summary = (
        pair_table.groupby(["subject", "pair_name"], as_index=False)
        .agg(
            distance_m=("distance_m", "first"),
            is_ss=("is_ss", "first"),
            is_ls=("is_ls", "first"),
        )
    )

    pair_summary["group"] = np.select(
        [pair_summary["is_ss"], pair_summary["is_ls"]],
        ["SS", "LS"],
        default="MID"
    )

    pair_rows.append(pair_summary)

    group_counts = pair_summary["group"].value_counts()

    subject_summary_rows.append({
        "subject": subject,
        "file": str(clean_file),
        "sfreq": raw_rest.info["sfreq"],
        "duration_s": raw_rest.times[-1],
        "n_fnirs_channels": len(picks_fnirs),
        "n_cw_channels": len(picks_cw),
        "distance_min_m": float(dists.min()),
        "distance_max_m": float(dists.max()),
        "n_ss_channels": int(ss_mask.sum()),
        "n_ls_channels": int(ls_mask.sum()),
        "n_pairs_total": len(pair_summary),
        "n_pairs_ss": int(group_counts.get("SS", 0)),
        "n_pairs_ls": int(group_counts.get("LS", 0)),
        "n_pairs_mid": int(group_counts.get("MID", 0)),
    })

subject_summary_df = pd.DataFrame(subject_summary_rows)
pair_summary_df = pd.concat(pair_rows, ignore_index=True)

subject_summary_df

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj100/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj101/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj102/resting_clean.snirf


/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)
/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_re

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj103/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj104/resting_clean.snirf


/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)
/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_re

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj86/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj91/resting_clean.snirf


...

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj97/resting_clean.snirf
Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj98/resting_clean.snirf


/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)
/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_re

Loading /home/asunkari/fnirs-representation-learning/snirf_dataset_2/Subj99/resting_clean.snirf


/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_rest = mne.io.read_raw_snirf(clean_file, preload=False)
/tmp/ipykernel_342674/3600149802.py:8: RuntimeWarning: The data only contains 2D location information for the optode positions. It is highly recommended that data is used which contains 3D location information for the optode positions. With only 2D locations it can not be guaranteed that MNE functions will behave correctly and produce accurate results. If it is not possible to include 3D positions in your data, please consider using the set_montage() function.
  raw_re

,subject,file,sfreq,duration_s,n_fnirs_channels,n_cw_channels,distance_min_m,distance_max_m,n_ss_channels,n_ls_channels,n_pairs_total,n_pairs_ss,n_pairs_ls,n_pairs_mid
0,Subj100,/home/asunkari/fnirs-representation-learning/s...,50.0,735.98,112,112,0.008,0.030463,16,96,56,8,48,0
1,Subj101,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
2,Subj102,/home/asunkari/fnirs-representation-learning/s...,50.0,731.98,112,112,0.008,0.030463,16,96,56,8,48,0
3,Subj103,/home/asunkari/fnirs-representation-learning/s...,50.0,716.98,112,112,0.008,0.030463,16,96,56,8,48,0
4,Subj104,/home/asunkari/fnirs-representation-learning/s...,50.0,669.98,112,112,0.008,0.030463,16,96,56,8,48,0
5,Subj86,/home/asunkari/fnirs-representation-learning/s...,50.0,590.98,112,112,0.008,0.030463,16,96,56,8,48,0
6,Subj91,/home/asunkari/fnirs-representation-learning/s...,50.0,641.98,112,112,0.008,0.030463,16,96,56,8,48,0
7,Subj92,/home/asunkari/fnirs-representation-learning/s...,50.0,588.98,112,112,0.008,0.030463,16,96,56,8,48,0
8,Subj94,/home/asunkari/fnirs-representation-learning/s...,50.0,692.98,112,112,0.008,0.030463,16,96,56,8,48,0
9,Subj95,/home/asunkari/fnirs-representation-learning/s...,50.0,765.98,112,112,0.008,0.030463,16,96,56,8,48,0
